In [2]:
import sys
sys.dont_write_bytecode = True
sys.path.insert(0, "..")
import numpy as np
from tqdm import tqdm
import pandas as pd
import datetime as dt
from scipy.stats import pearsonr
import pickle
import torch
import torch.nn as nn
import os
import time
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import math
from math import sqrt
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import codes.mnn_Utils as mnn
from codes.make_dataset import DatasetHist
from codes.data_utils import ips_omni_processor
import random

from codes.plot_utils import plot_losses

from codes.train_utils import cleanup

from termcolor import colored


# device = torch.device("cuda")
device = torch.device("xpu")

%load_ext autoreload
%autoreload 2

In [3]:
? ips_omni_processor

Init signature:  ips_omni_processor(ips_path, omni_path, sun_spot_path)
Docstring:     
A processor for constructing training datasets by combining IPS (Interplanetary Scintillation)
measurements, OMNI solar-wind data, and daily sunspot numbers.

This class loads, cleans, aligns, and time-calibrates IPS and OMNI data, merges them
with sun-spot information, and provides utilities for extracting structured inputs
and forecasting targets for machine-learning pipelines.

The processor performs the following high-level steps:

1. Reads IPS files named "VLIST<yy>" for years 1983–2024, merges them,
   converts dates, cleans velocity/error formats, and calibrates timestamps.

2. Loads curated OMNI solar-wind data, formats timestamps into the same time scale
   used for IPS data, and extracts smoothed solar-wind speed.

3. Loads daily sun-spot totals, cleans missing values, and merges the relevant
   time span with IPS data.

4. Produces a combined, time-normalized IPS dataset (`df_5`) with sel

In [4]:
ips_omni = ips_omni_processor(
    ips_path="../data/test_dwnld/",
    omni_path="../data/omni_avg_normalised_smoothed_extened.csv",
    sun_spot_path="../data/SN_d_tot_V2.0.csv")

IPS files found in folders: 
 ['VLIST83', 'VLIST84', 'VLIST85', 'VLIST86', 'VLIST87', 'VLIST88', 'VLIST89', 'VLIST90', 'VLIST91', 'VLIST92', 'VLIST93', 'VLIST94', 'VLIST95', 'VLIST96', 'VLIST97', 'VLIST98', 'VLIST99', 'VLIST00', 'VLIST01', 'VLIST02', 'VLIST03', 'VLIST04', 'VLIST05', 'VLIST06', 'VLIST07', 'VLIST08', 'VLIST09', 'VLIST10', 'VLIST11', 'VLIST12', 'VLIST13', 'VLIST14', 'VLIST15', 'VLIST16', 'VLIST17', 'VLIST18', 'VLIST19', 'VLIST20', 'VLIST21', 'VLIST22', 'VLIST23', 'VLIST24']
Making sun-spots data....
Making omni data ........
Taking sun spots data from one month prior to omni start date....
Formatting IPS data .........


/home/koba/Documents/ML/solar_weather/solar_wind/notebooks/../codes/data_utils.py:270: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.df_test_1 = pd.concat([self.df_test_1, df_test_0])


Shape of IPS data: (132077, 13)
(132077, 14)
Conditioning ips data .........
OMNI start date is calibrated to 200.02282467
Columns to be scaled except 'time': Index(['index', 'dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er',
       'sc_indx', 'time', 'day_total'],
      dtype='object')


In [6]:
omni_df = ips_omni.omni_df.copy()

In [9]:
omni_vls = omni_df.values

In [11]:
omni_df.head()

,swSpeed_Smth_0,time
0,0.6710416667,200.0228246651
1,0.6711538462,200.0245330981
2,0.6693750000,200.0262415311
3,0.6640000000,200.0279499641
4,0.6608593750,200.0296583971


In [17]:
np.hstack([omni_vls[:5], np.zeros((5, 16))]).shape

(5, 18)

In [13]:
omni_vls.shape

(231804, 2)

In [80]:
omni_vls_extd = np.hstack([omni_vls, np.zeros((omni_vls.shape[0], 16))])

In [82]:
omni_vls_extd.shape, omni_vls_extd[:5], omni_vls_extd

((231804, 18),
 array([[  0.67104167, 200.02282467,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ],
        [  0.67115385, 200.0245331 ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ],
        [  0.669375  , 200.02624153,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ,   0.        ,   0.        ,
           0.        ,   0.        ],
        [  0.664     , 200.02794996,   0.        ,   0.        ,
           0.        ,   0

In [72]:
omni_vls[:12], omni_vls[:6, 0].mean(), omni_vls[6:12, 0].mean()

(array([[  0.67104167, 200.02282467],
        [  0.67115385, 200.0245331 ],
        [  0.669375  , 200.02624153],
        [  0.664     , 200.02794996],
        [  0.66085938, 200.0296584 ],
        [  0.65830882, 200.03136683],
        [  0.65541667, 200.03307526],
        [  0.65375   , 200.0347837 ],
        [  0.6526875 , 200.03649213],
        [  0.65375   , 200.03820056],
        [  0.65630682, 200.039909  ],
        [  0.65771739, 200.04161743]]),
 np.float64(0.6657897852249876),
 np.float64(0.6549380626921388))

In [73]:
omni_vls[:5], omni_vls[1, 1]

(array([[  0.67104167, 200.02282467],
        [  0.67115385, 200.0245331 ],
        [  0.669375  , 200.02624153],
        [  0.664     , 200.02794996],
        [  0.66085938, 200.0296584 ]]),
 np.float64(200.02453309807083))

In [93]:
omni_vls[1:1+6], omni_vls[1:1+6, 0].mean()

(array([[  0.67115385, 200.0245331 ],
        [  0.669375  , 200.02624153],
        [  0.664     , 200.02794996],
        [  0.66085938, 200.0296584 ],
        [  0.65830882, 200.03136683],
        [  0.65541667, 200.03307526]]),
 np.float64(0.6631856185583209))

In [75]:
omni_vls_extd.shape

(231804, 18)

In [83]:
for i in range(omni_vls_extd.shape[0] - 96):
    for j in range(2, 18):
        omni_vls_extd[i, j] = omni_vls_extd[i + 6*(j-2) : i + 6*(j-1), 0].mean()

In [90]:
 6*(17-1)

96

In [66]:
for i in range(2):
    for j in range(2, 18):
        print(j, i + 6*(j-2), i + 6*(j-1))
    print("break", i)

2 0 6
3 6 12
4 12 18
5 18 24
6 24 30
7 30 36
8 36 42
9 42 48
10 48 54
11 54 60
12 60 66
13 66 72
14 72 78
15 78 84
16 84 90
17 90 96
break 0
2 1 7
3 7 13
4 13 19
5 19 25
6 25 31
7 31 37
8 37 43
9 43 49
10 49 55
11 55 61
12 61 67
13 67 73
14 73 79
15 79 85
16 85 91
17 91 97
break 1


In [92]:
omni_vls_extd[1,2]

np.float64(0.6631856185583209)

In [95]:
omni_df.columns

Index(['swSpeed_Smth_0', 'time'], dtype='object')

In [96]:
column_names = [f"target_{i}" for i in range(16)]
column_names = ["vel", "time"] + column_names
column_names

['vel',
 'time',
 'target_0',
 'target_1',
 'target_2',
 'target_3',
 'target_4',
 'target_5',
 'target_6',
 'target_7',
 'target_8',
 'target_9',
 'target_10',
 'target_11',
 'target_12',
 'target_13',
 'target_14',
 'target_15']

In [97]:
omni_extnd_df = pd.DataFrame(omni_vls_extd, columns=column_names)

In [103]:
omni_extnd_df[omni_extnd_df.target_0 > 0.0]

,vel,time,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,target_9,target_10,target_11,target_12,target_13,target_14,target_15
0,0.6710416667,200.0228246651,0.6657897852,0.6549380627,0.6627083333,0.6636197917,0.6651822917,0.6670920139,0.6404427083,0.6052430556,0.5750781250,0.5494791667,0.5350781250,0.5255729167,0.5132031250,0.5023263889,0.4944184028,0.4846180556
1,0.6711538462,200.0245330981,0.6631856186,0.6554936182,0.6634375000,0.6636892361,0.6662065972,0.6646875000,0.6345920139,0.5996093750,0.5705208333,0.5461979167,0.5334461806,0.5237500000,0.5111197917,0.5009288194,0.4929166667,0.4834288194
2,0.6693750000,200.0262415311,0.6602849775,0.6568043821,0.6637065972,0.6636284722,0.6674739583,0.6611805556,0.6287065972,0.5942187500,0.5659722222,0.5434895833,0.5317274306,0.5218229167,0.5092013889,0.4994878472,0.4913541667,0.4824218750
3,0.6640000000,200.0279499641,0.6575037275,0.6586307710,0.6636979167,0.6636197917,0.6686631944,0.6566232639,0.6228125000,0.5890364583,0.5615104167,0.5411979167,0.5300347222,0.5198350694,0.5074218750,0.4980555556,0.4897135417,0.4815451389
4,0.6608593750,200.0296583971,0.6557953942,0.6603321599,0.6636545139,0.6639843750,0.6689670139,0.6515972222,0.6168489583,0.5842274306,0.5572569444,0.5389756944,0.5285677083,0.5176215278,0.5056684028,0.4967881944,0.4879513889,0.4807465278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231703,0.4804166667,595.8718791413,0.4836024306,0.5062239583,0.5254861111,0.5323177083,0.5267708333,0.5162532682,0.5184397981,0.5148357290,0.5109527884,0.5130569540,0.5107223471,0.4955729167,0.4714843750,0.4507812500,0.4386197917,0.4402256944
231704,0.4812500000,595.8735875744,0.4859809028,0.5103125000,0.5275520833,0.5324131944,0.5245138889,0.5159138446,0.5188433222,0.5134543636,0.5111890403,0.5132450858,0.5091846727,0.4917013889,0.4677343750,0.4480468750,0.4378125000,0.4417274306
231705,0.4824479167,595.8752960074,0.4891319444,0.5139149306,0.5292881944,0.5320312500,0.5223241951,0.5160362031,0.5188975304,0.5122474943,0.5115325514,0.5132638067,0.5071639648,0.4878038194,0.4639322917,0.4456944444,0.4374479167,0.4433246528
231706,0.4833854167,595.8770044404,0.4928385417,0.5172916667,0.5305989583,0.5312673611,0.5202308754,0.5164851607,0.5184672399,0.5113892869,0.5119372100,0.5130670049,0.5047674404,0.4837934028,0.4604253472,0.4435156250,0.4374045139,0.4450607639


In [104]:
ips_omni.omni_df

,swSpeed_Smth_0,time
0,0.6710416667,200.0228246651
1,0.6711538462,200.0245330981
2,0.6693750000,200.0262415311
3,0.6640000000,200.0279499641
4,0.6608593750,200.0296583971
...,...,...
231799,0.4460937500,596.0358887106
231800,0.4477083333,596.0375971436
231801,0.4494791667,596.0393055767
231802,0.4515625000,596.0410140097


In [109]:
omni_extnd_df.iloc[0,2:].values.shape

(16,)

In [110]:
missing_df = pd.read_csv("../data/data_generated/missing.csv")

In [111]:
missing_corrected_df = pd.read_csv("../data/data_generated/missing_corrected.csv")

In [114]:
missing_corrected_df.describe()

,Unnamed: 0,id,missed
count,16665.0000000000,16665.0000000000,16665.0000000000
mean,8332.0000000000,64222.8366036604,25.0110411041
std,4810.9154534246,18945.5457358705,3.4875986353
min,0.0000000000,3109.0000000000,20.0000000000
25%,4166.0000000000,61217.0000000000,22.0000000000
50%,8332.0000000000,70635.0000000000,25.0000000000
75%,12498.0000000000,76746.0000000000,28.0000000000
max,16664.0000000000,81210.0000000000,31.0000000000


In [115]:
missing_df.describe()

,id,missed
count,16668.0000000000,16668.0000000000
mean,64222.5101391889,25.0115790737
std,18948.3772931899,3.4881861426
min,3109.0000000000,20.0000000000
25%,61217.7500000000,22.0000000000
50%,70635.5000000000,25.0000000000
75%,76747.2500000000,28.0000000000
max,81211.0000000000,31.0000000000


In [119]:
full_df = pd.read_csv("../data/data_generated/full_df.csv")

In [120]:
full_corrected_df = pd.read_csv("../data/data_generated/full_corrected_df.csv")

In [121]:
full_df

,idx,X_dist_0,X_hla_0,X_hlo_0,X_gla_0,X_glo_0,X_carr_0,X_v_0,X_er_0,X_sc_indx_0,...,y_swSpeed_Smth_0_6,y_swSpeed_Smth_0_7,y_swSpeed_Smth_0_8,y_swSpeed_Smth_0_9,y_swSpeed_Smth_0_10,y_swSpeed_Smth_0_11,y_swSpeed_Smth_0_12,y_swSpeed_Smth_0_13,y_swSpeed_Smth_0_14,y_swSpeed_Smth_0_15
0,0,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,0.0061935484,...,0.6554166667,0.6537500000,0.6526875000,0.6537500000,0.6563068182,0.6577173913,0.6587500000,0.6616145833,0.6636458333,0.6639583333
1,1,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,0.0061935484,...,0.6537500000,0.6526875000,0.6537500000,0.6563068182,0.6577173913,0.6587500000,0.6616145833,0.6636458333,0.6639583333,0.6644791667
2,2,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,0.0061935484,...,0.6526875000,0.6537500000,0.6563068182,0.6577173913,0.6587500000,0.6616145833,0.6636458333,0.6639583333,0.6644791667,0.6638020833
3,3,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,0.0061935484,...,0.6537500000,0.6563068182,0.6577173913,0.6587500000,0.6616145833,0.6636458333,0.6639583333,0.6644791667,0.6638020833,0.6631250000
4,4,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,0.0061935484,...,0.6563068182,0.6577173913,0.6587500000,0.6616145833,0.6636458333,0.6639583333,0.6644791667,0.6638020833,0.6631250000,0.6632291667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81207,81207,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.5001562500,0.4993229167,0.4991666667,0.4981250000,0.4972395833,0.4945833333,0.4920833333,0.4893229167,0.4870312500,0.4835416667
81208,81208,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4993229167,0.4991666667,0.4981250000,0.4972395833,0.4945833333,0.4920833333,0.4893229167,0.4870312500,0.4835416667,0.4810416667
81209,81209,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4991666667,0.4981250000,0.4972395833,0.4945833333,0.4920833333,0.4893229167,0.4870312500,0.4835416667,0.4810416667,0.4781250000
81210,81210,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4981250000,0.4972395833,0.4945833333,0.4920833333,0.4893229167,0.4870312500,0.4835416667,0.4810416667,0.4781250000,0.4752083333


In [122]:
full_corrected_df

,Unnamed: 0,idx,X_dist_0,X_hla_0,X_hlo_0,X_gla_0,X_glo_0,X_carr_0,X_v_0,X_er_0,...,y_swSpeed_Smth_0_6,y_swSpeed_Smth_0_7,y_swSpeed_Smth_0_8,y_swSpeed_Smth_0_9,y_swSpeed_Smth_0_10,y_swSpeed_Smth_0_11,y_swSpeed_Smth_0_12,y_swSpeed_Smth_0_13,y_swSpeed_Smth_0_14,y_swSpeed_Smth_0_15
0,0,0,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,...,0.6404427083,0.6052430556,0.5750781250,0.5494791667,0.5350781250,0.5255729167,0.5132031250,0.5023263889,0.4944184028,0.4846180556
1,1,1,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,...,0.6345920139,0.5996093750,0.5705208333,0.5461979167,0.5334461806,0.5237500000,0.5111197917,0.5009288194,0.4929166667,0.4834288194
2,2,2,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,...,0.6287065972,0.5942187500,0.5659722222,0.5434895833,0.5317274306,0.5218229167,0.5092013889,0.4994878472,0.4913541667,0.4824218750
3,3,3,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,...,0.6228125000,0.5890364583,0.5615104167,0.5411979167,0.5300347222,0.5198350694,0.5074218750,0.4980555556,0.4897135417,0.4815451389
4,4,4,0.3018867925,0.8657718121,0.2052980132,0.8471337580,0.6166666667,0.0026315789,0.3129855716,0.7755102041,...,0.6168489583,0.5842274306,0.5572569444,0.5389756944,0.5285677083,0.5176215278,0.5056684028,0.4967881944,0.4879513889,0.4807465278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81206,81206,81206,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,...,0.5061631944,0.5317447917,0.5518750000,0.5537065972,0.5357725694,0.5219791667,0.5052604167,0.4877864583,0.4763020833,0.4717615991
81207,81207,81207,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,...,0.5105815972,0.5354079861,0.5548871528,0.5505902778,0.5335763889,0.5192100694,0.5021354167,0.4855815972,0.4747482639,0.4720546613
81208,81208,81208,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,...,0.5149131944,0.5388454861,0.5569270833,0.5472395833,0.5314583333,0.5166059028,0.4989236111,0.4835503472,0.4734848960,0.4724933764
81209,81209,81209,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,...,0.5192708333,0.5420572917,0.5578385417,0.5438715278,0.5294097222,0.5140190972,0.4957465278,0.4816840278,0.4725288799,0.4730798457
